# Ein LLM als Klassifikator

Sechs Skripte haben das Text Mining von Grund auf aufgebaut: Zählen, Annotation, Vergleich, unüberwachte Strukturen, überwachte Klassifikation, Embeddings. Dieses letzte Skript fragt, was ein **großes Sprachmodell (LLM)** daran ändert. Dazu geben wir ihm genau die Aufgabe aus Skript 05 und 06 - einer Schlagzeile das Ressort des ORF zuordnen - auf genau denselben Testdaten und vergleichen es mit allem, was wir bisher gebaut haben.

Ein LLM unterscheidet sich von den bisherigen Modellen grundsätzlich, nicht nur in der Größe:

- Es ist **generativ**: Es erzeugt Text als Antwort auf Anweisungen. Eine Klassifikation wird also *erfragt*, nicht trainiert.
- Es braucht zu Beginn **keine Trainingsdaten**. Gelabelte Daten braucht es trotzdem, um es zu *evaluieren* - der Schritt, der am häufigsten ausgelassen wird.
- Es läuft auf fremden Rechnern. Jeder Aufruf **kostet Geld**, schickt den Text an einen Anbieter und hängt von einer Modellversion ab, auf die wir keinen Einfluss haben.

Am Ende dieses Skripts kannst du:

- ein LLM aus Python mit einem Schema für **strukturierte Ausgabe** aufrufen, sodass es Labels liefert, die ein Programm weiterverarbeiten kann, statt Fließtext;
- **Zero-Shot**- und **Few-Shot**-Prompts schreiben und messen, wie viel der Prompt ausmacht;
- einen LLM-Klassifikator **gegen dieselbe Baseline** evaluieren wie jedes andere Modell;
- Accuracy gegen Kosten, Geschwindigkeit, Reproduzierbarkeit und Datenschutz abwägen.

### Ausführen dieses Skripts

Die Aufrufe gehen über das Python-SDK `anthropic` an die API von Anthropic. Dafür brauchst du einen API-Schlüssel aus der [Claude Console](https://platform.claude.com/), der vor dem Start von Jupyter als Umgebungsvariable `ANTHROPIC_API_KEY` gesetzt sein muss. **Das Skript ruft die API nur auf, wenn du in der folgenden Zelle `RUN_LLM = True` setzt.** Ist der Schalter aus, laufen trotzdem alle Zellen: Prompt und Kostenschätzung werden angezeigt, und die Evaluation verwendet gespeicherte Ergebnisse, falls ein früherer Lauf welche erzeugt hat.

Ein vollständiger Lauf klassifiziert die gut 400 Schlagzeilen der Testdaten zweimal (zwei Prompt-Varianten), in Batches. Zum Zeitpunkt des Schreibens kostet das in der Größenordnung von einem US-Dollar; der Abschnitt "Die Kosten vorab abschätzen" gibt eine Schätzung über den kostenlosen Endpunkt zum Zählen von Tokens aus, bevor etwas ausgegeben wird. Die Ergebnisse werden in `_cache/` gespeichert, sodass ein zweiter Lauf des Skripts sie von dort liest, statt erneut zu bezahlen.

In [ ]:
RUN_LLM = False  # Setze auf True, um die API aufzurufen (kostet Geld - siehe oben)

## Präambel

Wir laden alle Pakete und Module, die wir im Folgenden benötigen.

In [ ]:
# Standardbibliothek
import json
import os
import time

# Datentransformation
import numpy as np
import pandas as pd

# Client für die API von Anthropic
import anthropic

# Pakete für Textklassifikation und Evaluation
from sentence_transformers import SentenceTransformer
from sklearn.dummy import DummyClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, f1_score
from sklearn.pipeline import FeatureUnion, make_pipeline
from sklearn.svm import LinearSVC

# Hilfsfunktionen für den Text-Mining-Teil des Kurses
import textmining as tm

# Darstellungsoptionen für Tabellen und Grafiken
pd.set_option("display.max_colwidth", 90)

## Daten

Wir verwenden dieselbe Aufgabe wie in Skript 05 und 06: Jeder Schlagzeile soll ihr ORF-Ressort zugeordnet werden. `tm.subject_task()` liefert dieselbe zeitliche Aufteilung in Trainings- und Testdaten, sodass alle Modelle auf denselben Schlagzeilen bewertet werden.

In [ ]:
# Lade die Trainings- und Testdaten der Ressort-Aufgabe
train, test = tm.subject_task()
# Die sechs Labels: fünf Ressorts und "Sonstiges"
LABELS = tm.SUBJECTS + [tm.OTHER]
print(f"Testdaten: {len(test)} Schlagzeilen, Labels: {LABELS}")

## Der Client

`anthropic.Anthropic()` liest den API-Schlüssel aus der Umgebung. Als Modell verwenden wir Claude Opus 5.5 (`claude-opus-5-5`).

In [ ]:
MODEL = "claude-opus-5-5"
HAVE_KEY = bool(os.environ.get("ANTHROPIC_API_KEY") or os.environ.get("ANTHROPIC_AUTH_TOKEN"))
# Lege den Client nur an, wenn ein API-Schlüssel gefunden wurde
client = anthropic.Anthropic() if HAVE_KEY else None
print("Zugangsdaten gefunden" if HAVE_KEY else "kein ANTHROPIC_API_KEY gesetzt - API-Aufrufe werden übersprungen")
if RUN_LLM and not HAVE_KEY:
    raise RuntimeError("RUN_LLM ist True, aber es ist kein API-Schlüssel gesetzt. Siehe 'Ausführen dieses Skripts' oben.")

## Zwei Prompts

Im Mittelpunkt dieses Skripts steht der Vergleich zweier Prompts.

**Naiver Zero-Shot-Prompt**: die Labels nennen und fragen. So sehen die meisten ersten Versuche aus.

**Informierter Few-Shot-Prompt**: beschreiben, was die Labels tatsächlich *bedeuten* - mit dem, was Skript 05 und 06 über sie herausgefunden haben - und einige gelabelte Beispiele zeigen. Die Fehleranalyse in Skript 05 hat gezeigt, dass die Ressorts des ORF die **Redaktion** benennen, nicht den Ort des Geschehens: Ein Hurrikan in Florida ist *Chronik*, deutsche Politik ist *Ausland*. Ein Modell, dem nur die Namen der Labels genannt werden, kann das nicht wissen.

Die Few-Shot-Beispiele stammen aus den **Trainingsdaten**. Würden wir sie aus den Testdaten nehmen, gelangten Antworten in den Prompt - Data Leakage, genau wie die zufällige Aufteilung in Skript 05 Meldungen durchsickern ließ.

Die Prompts selbst sind auf Englisch formuliert. Wir lassen sie unverändert, weil sie Teil des Experiments sind.

In [ ]:
# Naiver Prompt: nur die Namen der Labels
NAIVE_SYSTEM = (
    "You classify headlines from the Austrian news broadcaster ORF into subject "
    f"categories. The categories are: {', '.join(LABELS)}."
)

# Definitionen der Ressorts für den informierten Prompt
DEFINITIONS = {
    "Ausland": "Foreign politics and international affairs: governments, elections, wars and diplomacy outside Austria. "
               "German politics counts as Ausland, since ORF is Austrian.",
    "Inland": "Austrian domestic politics: the federal and state governments, parties, parliament, "
              "ministers, and studio interviews with Austrian politicians and experts.",
    "Wirtschaft": "Business and the economy, in Austria or abroad: companies, markets, prices, inflation, "
                  "jobs, economic policy figures.",
    "Chronik": "Incidents and human-interest news, in Austria or abroad: crime, accidents, disasters, "
               "weather events, court cases about individuals.",
    "Kultur": "Culture: film, music, literature, art, festivals, awards, deaths of artists.",
    "Sonstiges": "Everything else: science, technology and AI, health, religion, media, lifestyle, "
                 "lottery results.",
}

# Ziehe drei Schlagzeilen pro Label aus den Trainingsdaten und mische sie, damit die Labels nicht in Blöcken erscheinen
examples = pd.concat([g.sample(3, random_state=0) for _, g in train.groupby("label")]).sample(frac=1, random_state=0)

# Informierter Prompt: Erklärung, Definitionen und Beispiele
INFORMED_SYSTEM = (
    "You classify headlines from the Austrian news broadcaster ORF into the subject "
    "categories ORF's own editors use. The categories name the editorial desk that "
    "covers a story, not the place where it happened.\n\n"
    + "\n".join(f"- {label}: {text}" for label, text in DEFINITIONS.items())
    + "\n\nExamples of headlines and the category ORF assigned:\n"
    + "\n".join(f"- {r.title} -> {r.label}" for r in examples.itertuples())
)
# Zeige den informierten Prompt
print(INFORMED_SYSTEM)

## Strukturierte Ausgabe

In natürlicher Sprache gefragt, antwortet ein LLM in natürlicher Sprache: *"Diese Schlagzeile handelt höchstwahrscheinlich von Außenpolitik, also würde ich Ausland sagen."* Ein Programm kann das nicht zuverlässig verwenden. **Strukturierte Ausgaben** beschränken die Antwort auf ein JSON-Schema, sodass die API gültiges JSON in der festgelegten Form garantiert - hier ein Label pro Schlagzeilen-ID, wobei das Label über ein `enum` auf die sechs zulässigen Werte beschränkt ist. Die Schlagzeilen werden nummeriert und in Batches geschickt, sodass eine Anfrage viele davon klassifiziert.

In [ ]:
# JSON-Schema: eine Liste aus ID und Label, das Label nur aus LABELS
SCHEMA = {
    "type": "object",
    "properties": {
        "labels": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {"id": {"type": "integer"}, "label": {"type": "string", "enum": LABELS}},
                "required": ["id", "label"],
                "additionalProperties": False,
            },
        }
    },
    "required": ["labels"],
    "additionalProperties": False,
}


# Baue die Nachricht für einen Batch nummerierter Schlagzeilen
def batch_message(titles: list[str]) -> str:
    lines = "\n".join(f"{i}. {t}" for i, t in enumerate(titles))
    return f"Classify each of these {len(titles)} headlines. Return one label per id.\n\n{lines}"


# Zeige die Nachricht für die ersten drei Schlagzeilen
print(batch_message(test["title"].head(3).tolist()))

Die Anfrage selbst, mit drei Einstellungen, die eine Erklärung verdienen:

- **`effort="low"`** - Claude Opus 5.5 denkt vor der Antwort immer nach, und `effort` legt fest, wie viel. Die Klassifikation kurzer Texte braucht wenig; niedriger Aufwand ist günstiger und schneller.
- **`fallbacks="default"`** - Lehnen die Sicherheitsklassifikatoren des Modells eine Anfrage ab (Nachrichten über Gewalt oder Kriminalität können sie gelegentlich auslösen), wiederholt die API die Anfrage mit einem geeigneten Ausweichmodell, statt eine Ablehnung zurückzugeben. Eine Ablehnung kann trotzdem vorkommen, deshalb prüft der Code `stop_reason`, bevor er die Antwort liest.
- **`cache_control`** am System-Prompt - alle Batches teilen ihn, daher kann er zwischengespeichert und zu einem Bruchteil des Preises erneut gelesen werden. Prompts unter einer Mindestlänge werden stillschweigend nicht gecacht; `usage.cache_read_input_tokens` zeigt, ob es funktioniert hat.

In [ ]:
# Schicke einen Batch an die API, gib die Labels und den Token-Verbrauch zurück
def classify_batch(titles: list[str], system: str) -> tuple[list[str | None], dict]:
    response = client.beta.messages.create(
        model=MODEL,
        max_tokens=8000,
        betas=["server-side-fallback-2026-07-01"],
        fallbacks="default",
        output_config={"effort": "low", "format": {"type": "json_schema", "schema": SCHEMA}},
        system=[{"type": "text", "text": system, "cache_control": {"type": "ephemeral"}}],
        messages=[{"role": "user", "content": batch_message(titles)}],
    )
    usage = {
        "input": response.usage.input_tokens,
        "output": response.usage.output_tokens,
        "cache_read": response.usage.cache_read_input_tokens or 0,
        "cache_write": response.usage.cache_creation_input_tokens or 0,
    }
    if response.stop_reason != "end_turn":
        # Eine Ablehnung oder eine abgeschnittene Antwort: Verbuche den Batch als
        # nicht klassifiziert, statt zu raten, und weise darauf hin.
        print(f"  Batch nicht klassifiziert: stop_reason={response.stop_reason}")
        return [None] * len(titles), usage
    # Lies das JSON aus dem letzten Textblock und ordne die Labels den IDs zu
    text = [block.text for block in response.content if block.type == "text"][-1]
    by_id = {item["id"]: item["label"] for item in json.loads(text)["labels"]}
    return [by_id.get(i) for i in range(len(titles))], usage

## Die Kosten vorab abschätzen

Der Endpunkt zum Zählen von Tokens ist kostenlos, daher lassen sich die Kosten eines Laufs abschätzen, bevor wir dafür bezahlen. Die Preise gelten pro Million Tokens und waren zum Zeitpunkt des Schreibens dieses Skripts korrekt; prüfe die aktuelle Preisliste, bevor du dich darauf verlässt.

In [ ]:
PRICE_IN, PRICE_OUT, PRICE_CACHE_READ = 4.00, 20.00, 0.20  # USD pro Million Tokens, claude-opus-5-5
# Teile die Testdaten in Batches zu je BATCH Schlagzeilen
BATCH = 40
batches = [test["title"].iloc[i : i + BATCH].tolist() for i in range(0, len(test), BATCH)]

if client is not None:
    # Zähle die Input-Tokens des ersten Batches (kostenlos)
    one = client.messages.count_tokens(
        model=MODEL, system=INFORMED_SYSTEM, messages=[{"role": "user", "content": batch_message(batches[0])}]
    ).input_tokens
    # Output: grob 15 Tokens pro Label plus Reasoning; großzügig bemessen
    est_out = BATCH * 15 + 1500
    per_variant = len(batches) * (one * PRICE_IN + est_out * PRICE_OUT) / 1e6
    print(f"{len(batches)} Batches pro Prompt, je ~{one:,} Input-Tokens")
    print(f"geschätzte Kosten: ~${per_variant:.2f} pro Prompt-Variante, ~${2 * per_variant:.2f} für beide")
else:
    print(f"{len(batches)} Batches mit bis zu {BATCH} Schlagzeilen pro Prompt-Variante (kein Schlüssel: Schätzung übersprungen)")

## Die Klassifikation ausführen

Die Ergebnisse werden pro Prompt-Variante in `_cache/` geschrieben. Existiert die Datei bereits, wird sie gelesen und keine Anfrage gesendet.

In [ ]:
def run_variant(name: str, system: str) -> pd.DataFrame | None:
    # Gespeicherte Ergebnisse haben Vorrang - dann wird nichts gesendet
    path = tm.CACHE / f"llm-{name}-{MODEL}-{len(test)}.parquet"
    if path.exists():
        print(f"{name}: lese gespeicherte Ergebnisse aus {path.name}")
        return pd.read_parquet(path)
    if not RUN_LLM:
        print(f"{name}: keine gespeicherten Ergebnisse und RUN_LLM ist False - übersprungen")
        return None
    # Klassifiziere alle Batches nacheinander
    labels, usage, start = [], [], time.time()
    for k, titles in enumerate(batches):
        out, u = classify_batch(titles, system)
        labels += out
        usage.append(u)
        print(f"  {name}: Batch {k + 1}/{len(batches)}", end="\r")
    # Summiere den Token-Verbrauch und berechne die Kosten (Schreiben in den Cache kostet 25 % Aufschlag)
    u = pd.DataFrame(usage).sum()
    cost = (u["input"] * PRICE_IN + u["cache_read"] * PRICE_CACHE_READ + u["cache_write"] * PRICE_IN * 1.25
            + u["output"] * PRICE_OUT) / 1e6
    print(f"\n{name}: {time.time() - start:.0f}s, Tokens rein {u['input']:,} (+{u['cache_read']:,} aus dem Cache) "
          f"raus {u['output']:,}, Kosten ~${cost:.2f}")
    # Speichere die Labels im Cache
    result = pd.DataFrame({"item_key": test["item_key"], "predicted": labels})
    tm.CACHE.mkdir(exist_ok=True)
    result.to_parquet(path)
    return result


# Klassifiziere die Testdaten mit beiden Prompts
llm = {name: run_variant(name, system) for name, system in [("naive", NAIVE_SYSTEM), ("informed", INFORMED_SYSTEM)]}

## Der Vergleich

Alle Modelle dieses Kurses, bewertet auf denselben gut 400 Schlagzeilen aus demselben zurückgehaltenen Zeitraum. Die Modelle aus Skript 05 und 06 werden hier neu trainiert, damit die Tabelle für sich allein steht. Logistische Regression und SVM kennen wir bereits aus dem Skript `03-data-mining-classification`.

In [ ]:
# Skript 05: TF-IDF auf Wörtern und Zeichen-N-Grammen + lineare SVM
tok = r"(?u)\b[^\W\d_][^\W\d_-]*(?:-[^\W\d_]+)*\b"
lesson05 = make_pipeline(
    FeatureUnion([
        ("words", TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True, token_pattern=tok)),
        ("chars", TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), min_df=2, sublinear_tf=True)),
    ]),
    LinearSVC(C=1),
).fit(train["title"], train["label"])

# Skript 06: Sentence-Transformer-Embeddings + logistische Regression
encoder = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")
emb = lambda s: encoder.encode(s.tolist(), normalize_embeddings=True, show_progress_bar=False)
lesson06 = LogisticRegression(max_iter=3000, C=4).fit(emb(train["title"]), train["label"])

# Vorhersagen aller Modelle auf den Testdaten
predictions = {
    "Baseline: häufigste Klasse": DummyClassifier(strategy="most_frequent").fit(train["title"], train["label"]).predict(test["title"]),
    "05: TF-IDF + SVM (trainiert)": lesson05.predict(test["title"]),
    "06: Embeddings + logistische Regression (trainiert)": lesson06.predict(emb(test["title"])),
}
# Ergänze die LLM-Ergebnisse, sofern vorhanden
for name, result in llm.items():
    if result is not None:
        predictions[f"07: LLM, Prompt {name} (ohne Training)"] = result["predicted"].fillna("ohne Label").to_numpy()

# Berechne Accuracy und Makro-F1 pro Modell
pd.DataFrame(
    {name: {"Accuracy": accuracy_score(test["label"], p), "Makro-F1": f1_score(test["label"], p, average="macro", labels=LABELS)}
     for name, p in predictions.items()}
).T.round(3)

Wenn du das LLM ausgeführt hast, lies die Tabelle in zwei Richtungen.

**Zwischen den beiden Prompt-Zeilen.** Das Modell ist dasselbe; nur die Beschreibung hat sich geändert. Ein Unterschied zwischen naivem und informiertem Prompt ist daher der Wert davon, *die eigenen Labels zu kennen* - und das Wissen im informierten Prompt, vor allem dass die Kategorien Redaktionen und keine Orte sind, stammt aus der Konfusionsmatrix in Skript 05, nicht vom LLM. Übung 1 prüft, welcher Teil des Prompts die Arbeit leistet.

**Über die Methoden hinweg.** Ob das LLM die trainierten Modelle schlägt und um wie viel, hängt von der Aufgabe ab. Bei der Klassifikation von Schlagzeilen bringt ein LLM breites Weltwissen mit - es weiß, wer Minister ist und in welchem Land eine Stadt liegt -, aber die Ablagekonventionen des ORF, die die trainierten Modelle aus tausend Beispielen gelernt haben, hat es nie gesehen. Wo diese Konventionen eigenwillig sind, behalten die trainierten Modelle einen Vorsprung; wo das Weltwissen entscheidet, gewinnt das LLM.

### Wo liegt das LLM falsch?

Analysiert wird der informierte Prompt; der naive wird nur verwendet, wenn er das einzige vorhandene Ergebnis ist.

In [ ]:
# Wähle die LLM-Zeile für die Fehleranalyse, bevorzugt den informierten Prompt
llm_rows = [k for k in predictions if k.startswith("07")]
chosen = next((k for k in llm_rows if "informed" in k), llm_rows[0] if llm_rows else None)
if chosen is None:
    print("Noch keine LLM-Ergebnisse - setze RUN_LLM = True und führe das Skript erneut aus, um diesen Abschnitt zu füllen.")
else:
    p = predictions[chosen]
    print(chosen)
    print(classification_report(test["label"], p, labels=LABELS, digits=2, zero_division=0))
    # Vergleiche mit dem Embedding-Modell aus Skript 06
    view = test.assign(llm=p, embedding=predictions["06: Embeddings + logistische Regression (trainiert)"])
    print("Schlagzeilen, die das LLM falsch und das Embedding-Modell richtig klassifiziert:")
    display(view[(view["llm"] != view["label"]) & (view["embedding"] == view["label"])][["title", "label", "llm"]].head(10))

## Die Abwägung

Ein Score ist nur die Hälfte des Vergleichs. Die andere Hälfte:

| | TF-IDF + SVM | Embeddings + LR | LLM über API |
|---|---|---|---|
| benötigte gelabelte Daten | ~1.000 zum Trainieren | ~1.000 zum Trainieren | keine für den Start, aber einige zur Evaluation |
| Kosten pro 1.000 Schlagzeilen | praktisch null | praktisch null | echtes Geld, jedes Mal |
| Geschwindigkeit | Tausende pro Sekunde | Hunderte pro Sekunde | Sekunden pro Batch |
| läuft offline | ja | ja, nach einem Download | nein |
| Text verlässt den eigenen Rechner | nein | nein | ja |
| gleiche Eingabe, gleiche Ausgabe | ja | ja | nicht garantiert |
| kann sich erklären | Gewichte pro Wort | kaum | in Worten - nicht immer wahrheitsgetreu |
| ändert sich ohne dein Zutun | nein | nein | wenn der Anbieter das Modell aktualisiert |

Daraus ergeben sich zwei praktische Muster:

- **LLM als Annotator.** Lass das LLM einmalig einige tausend Einträge labeln, prüfe eine Stichprobe von Hand, trainiere dann einen günstigen Klassifikator (Skript 05 oder 06) auf dem Ergebnis und setze *diesen* im großen Maßstab ein. Für das LLM bezahlst du nur einmal.
- **LLM für die schwierigen Fälle.** Lass ein günstiges Modell die sichere Mehrheit erledigen und schicke nur unsichere Einträge - solche mit niedriger vorhergesagter Wahrscheinlichkeit - an das LLM.

Und eine Warnung, die für jede Methode dieses Kurses gilt und für LLMs ganz besonders: **Die Ausgabe eines Modells ist erst dann eine Messung, wenn sie validiert wurde.** Die Gewohnheiten dieses Kurses - eine Baseline, zurückgehaltene Testdaten, das Lesen der Fehler - haben den Vergleich oben erst möglich gemacht. Sie werden wichtiger, nicht unwichtiger, wenn das Modell beeindruckend ist.

## Zusammenfassung

- Ein LLM kann Text ohne Training klassifizieren, wenn die Aufgabe gut beschrieben ist, und **strukturierte Ausgaben** machen seine Antworten maschinenlesbar.
- Der **Prompt ist Teil des Modells**: Zu beschreiben, was die Labels bedeuten, kann so viel ausmachen wie die Wahl des Modells.
- Ein LLM wird genau wie jeder andere Klassifikator evaluiert - gleiche Aufteilung, gleiche Metriken, gleiche Fehleranalyse -, und sein Score wird gegen Kosten, Geschwindigkeit, Datenschutz und Reproduzierbarkeit abgewogen.

## Übungen

1. **Prompt-Ablation.** Entferne die Few-Shot-Beispiele aus dem informierten Prompt, behalte aber die Definitionen. Welcher Teil hat die Arbeit geleistet?
2. **Reproduzierbarkeit.** Klassifiziere dieselben 40 Schlagzeilen dreimal. Wie viele Labels ändern sich zwischen den Läufen?
3. **LLM als Annotator.** Labele die *Trainingsdaten* mit dem informierten Prompt, trainiere den Embedding-Klassifikator aus Skript 06 auf den Labels des LLM statt auf denen des ORF und bewerte ihn auf den Testdaten. Wie nah kommt er an das Modell heran, das auf den echten Labels trainiert wurde?
4. **Über die Klassifikation hinaus.** Lass das Modell für jede Schlagzeile den Hauptakteur und das betroffene Land als strukturiertes JSON extrahieren. Vergleiche seine Akteure auf 30 Schlagzeilen mit den Subjekten aus dem Dependenzparsing in Skript 02.

## Quellen

* https://docs.anthropic.com/
* https://scikit-learn.org/stable/modules/feature_extraction.html#text-feature-extraction
* https://scikit-learn.org/stable/modules/svm.html
* https://scikit-learn.org/stable/modules/model_evaluation.html
* https://sbert.net/
* https://pandas.pydata.org/docs/